# Day3: Exploratory Analysis

このノートブックでは、正規化したRNA-seqデータを PCA とヒートマップで眺める方法を学びます。

## なぜ探索的解析をするのか

差次的発現解析に進む前に、まずデータ全体の構造を見ることが重要です。条件ごとにサンプルがまとまっているか、極端に離れたサンプルがないかを可視化で確認します。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

counts = pd.DataFrame({
    "gene": ["TP53", "MYC", "GAPDH", "ACTB", "IL6"],
    "control_1": [120, 300, 5000, 4200, 30],
    "control_2": [100, 280, 5100, 4000, 25],
    "treated_1": [300, 600, 5300, 4100, 200],
    "treated_2": [280, 650, 5200, 4300, 220],
}).set_index("gene")

library_size = counts.sum(axis=0)
cpm = counts.div(library_size, axis=1) * 1_000_000
log_cpm = np.log2(cpm + 1)
log_cpm

## PCA

PCA は、多次元の発現データを少数の軸に圧縮して、サンプル間の近さを見やすくする方法です。近いサンプルほど発現パターンが似ています。

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(log_cpm.T)
pca_df = pd.DataFrame(coords, columns=["PC1", "PC2"], index=log_cpm.columns)
pca_df["condition"] = ["control", "control", "treated", "treated"]
pca_df

In [ ]:
plt.figure(figsize=(5, 4))
colors = {"control": "#4C78A8", "treated": "#F58518"}
for sample, row in pca_df.iterrows():
    plt.scatter(row["PC1"], row["PC2"], color=colors[row["condition"]], s=80)
    plt.text(row["PC1"] + 0.02, row["PC2"] + 0.02, sample)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of log-CPM")
plt.tight_layout()
plt.show()

## ヒートマップ的に見る

ここでは `matplotlib` だけで簡単な発現ヒートマップを描きます。色の違いで、どの遺伝子がどのサンプルで高いかを直感的に見られます。

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(log_cpm.values, aspect="auto", cmap="viridis")
plt.colorbar(label="log2(CPM + 1)")
plt.xticks(range(len(log_cpm.columns)), log_cpm.columns, rotation=30, ha="right")
plt.yticks(range(len(log_cpm.index)), log_cpm.index)
plt.title("Gene expression heatmap")
plt.tight_layout()
plt.show()

## どう読むか

- PCAで control 同士、treated 同士が近ければ、条件差が反映されている可能性があります。
- ヒートマップで `IL6` が treated 側で高ければ、条件差を視覚的にも確認できます。
- もし1サンプルだけ極端に離れていれば、外れ値や実験上の問題も疑います。

## ここまでの要点

- 探索的解析は、差次的発現解析の前にデータを眺める工程
- PCA はサンプル間の近さを見る
- ヒートマップは遺伝子ごとの発現パターンを見る
- 次は統計的に「どの遺伝子が変化したか」を考える